In [ ]:
!pip install -q tensorflow pandas numpy matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# In Google Colab, upload the CSV using:
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
print(df.isnull().sum())

StopIteration: 

In [ ]:
# In Google Colab, upload the CSV using:
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
print(df.isnull().sum())

In [ ]:
DISTRICT = "COIMBATORE"
STATE = "TAMIL NADU"
WINDOW_SIZE = 7

# Standardize column names
df.columns = df.columns.str.strip()

# Find the district column
district_col = next(
    (c for c in df.columns
     if c.strip().lower() in ["district", "district name"]),
    None
)

if district_col is None:
    raise ValueError(
        "District column not found. Inspect df.columns and update the code."
    )

df[district_col] = df[district_col].astype(str).str.strip()

# Filter the selected district
district_df = df[
    df[district_col].str.upper() == DISTRICT
].copy()

if "state_name" in df.columns:
    district_df = district_df[
        district_df["state_name"].astype(str).str.upper()
        == STATE
    ]

if district_df.empty:
    raise ValueError(
        "District not found. Check the district and state names "
        "using df[district_col].unique()."
    )

# Support long-format or wide-format data
date_col = next(
    (c for c in district_df.columns
     if c.lower() in ["date", "observation date"]),
    None
)

rain_col = next(
    (c for c in district_df.columns
     if c.lower() in [
         "actual rainfall", "daily actual", "rainfall",
         "actual", "rainfall (mm)"
     ]),
    None
)

if date_col and rain_col:
    ts = district_df[[date_col, rain_col]].copy()
    ts.columns = ["Date", "Rainfall"]
    ts["Date"] = pd.to_datetime(ts["Date"], errors="coerce")
    ts["Rainfall"] = pd.to_numeric(
        ts["Rainfall"], errors="coerce"
    )

else:
    # Wide format: daily values stored in date/day columns
    id_cols = [
        c for c in district_df.columns
        if c.lower() in [
            "index", "state code", "state name",
            "district code", "district name", "month"
        ]
    ]

    value_cols = [
        c for c in district_df.columns if c not in id_cols
    ]

    wide = district_df.melt(
        id_vars=[c for c in id_cols if c in district_df.columns],
        value_vars=value_cols,
        var_name="Day",
        value_name="Rainfall"
    )

    # Adapt this section if the file uses named dates
    # rather than day numbers.
    month_col = next(
        (c for c in wide.columns if c.lower() == "month"),
        None
    )

    if month_col is None:
        raise ValueError(
            "Wide-format file detected. Inspect its columns and "
            "construct the Date field from year, month, and day."
        )

    wide["Day"] = pd.to_numeric(wide["Day"], errors="coerce")
    wide["Month"] = pd.to_numeric(wide[month_col], errors="coerce")

    ts = pd.DataFrame({
        "Date": pd.to_datetime(
            {
                "year": 2022,
                "month": wide["Month"],
                "day": wide["Day"]
            },
            errors="coerce"
        ),
        "Rainfall": pd.to_numeric(
            wide["Rainfall"], errors="coerce"
        )
    })

# Convert invalid/missing markers to NaN
ts["Rainfall"] = ts["Rainfall"].replace(
    [-999, -999.0, -999.9, -9999, -9999.0],
    np.nan
)

ts = ts.dropna(subset=["Date"])
ts = ts.sort_values("Date")

# Aggregate duplicate observations for a date
ts = ts.groupby("Date", as_index=False)["Rainfall"].mean()

# Create a complete daily calendar
ts = ts.set_index("Date").asfreq("D")

# Rainfall cannot be negative; inspect before applying this rule
ts.loc[ts["Rainfall"] < 0, "Rainfall"] = np.nan

# Interpolate short gaps; fill remaining gaps with zero only
# when missing values mean no recorded rainfall in this dataset.
ts["Rainfall"] = ts["Rainfall"].interpolate(
    method="time", limit=2, limit_area="inside"
)
ts["Rainfall"] = ts["Rainfall"].fillna(0.0)

print(ts.head())
print("Number of daily observations:", len(ts))
print("Remaining missing values:", ts["Rainfall"].isna().sum())

plt.figure(figsize=(12, 4))
plt.plot(ts.index, ts["Rainfall"])
plt.title(f"Daily Rainfall — {DISTRICT}")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
rainfall = ts["Rainfall"].values.reshape(-1, 1)

# Chronological split: first 80% for training, last 20% for testing
split_idx = int(len(rainfall) * 0.80)

if split_idx <= WINDOW_SIZE or len(rainfall) - split_idx < 1:
    raise ValueError("Not enough observations for the selected split.")

# Fit scaler on training data only
scaler = MinMaxScaler()
scaler.fit(rainfall[:split_idx])

scaled = scaler.transform(rainfall)

def create_sequences(data, window=7):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled, WINDOW_SIZE)

# Target positions correspond to original indices 7, 8, ...
target_positions = np.arange(WINDOW_SIZE, len(scaled))

train_mask = target_positions < split_idx
test_mask = target_positions >= split_idx

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

test_dates = ts.index[target_positions[test_mask]]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
model = Sequential([
    LSTM(
        50,
        activation="tanh",
        input_shape=(WINDOW_SIZE, 1)
    ),
    Dropout(0.2),
    Dense(25, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

model.summary()

In [ ]:
# Predict on the held-out chronological test set
pred_scaled = model.predict(X_test, verbose=0)

# Convert predictions and actual targets back to millimetres
y_pred = scaler.inverse_transform(pred_scaled).ravel()
y_actual = scaler.inverse_transform(y_test).ravel()

# Rainfall cannot be negative
y_pred = np.maximum(y_pred, 0)

mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

print(f"Test MAE:  {mae:.3f} mm")
print(f"Test RMSE: {rmse:.3f} mm")

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    test_dates,
    y_actual,
    label="Actual Rainfall",
    color="blue",
    linewidth=1.8
)

plt.plot(
    test_dates,
    y_pred,
    label="Predicted Rainfall",
    color="red",
    linestyle="--",
    linewidth=1.5
)

plt.title(f"Actual vs Predicted Rainfall — {DISTRICT}")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Display predictions for unseen dates in the test set
results = pd.DataFrame({
    "Date": test_dates,
    "Actual Rainfall (mm)": y_actual,
    "Predicted Rainfall (mm)": y_pred
})

print("Test-set predictions:")
print(results.tail(10).round(2).to_string(index=False))

# Forecast the next 7 days after the final observed date
last_window = scaled[-WINDOW_SIZE:].copy()
future_predictions = []

for _ in range(7):
    next_scaled = model.predict(
        last_window.reshape(1, WINDOW_SIZE, 1),
        verbose=0
    )

    next_mm = scaler.inverse_transform(next_scaled)[0, 0]
    next_mm = max(0.0, next_mm)

    future_predictions.append(next_mm)

    # Normalize the predicted value for the next iteration
    next_scaled_clipped = scaler.transform(
        np.array([[next_mm]])
    )

    last_window = np.vstack([
        last_window[1:],
        next_scaled_clipped
    ])

future_dates = pd.date_range(
    start=ts.index[-1] + pd.Timedelta(days=1),
    periods=7,
    freq="D"
)

forecast_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted Rainfall (mm)": future_predictions
})

print("\nSeven-day future rainfall forecast:")
print(forecast_df.round(2).to_string(index=False))

# Plot the forecast
plt.figure(figsize=(10, 4))
plt.plot(
    ts.index[-14:],
    ts["Rainfall"].iloc[-14:],
    marker="o",
    label="Observed Rainfall"
)
plt.plot(
    future_dates,
    future_predictions,
    marker="o",
    linestyle="--",
    color="red",
    label="Forecast Rainfall"
)
plt.title(f"Future Rainfall Forecast — {DISTRICT}")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()